In [1]:
import pandas as pd
import numpy as np
import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '../..')))
from seq2seq import *
import pickle

from sklearn import metrics

from sklearn.utils import shuffle
import keras
from keras.models import Sequential
from keras.layers import Dense, Dropout, Input
from sklearn.preprocessing import StandardScaler
from keras import backend as K
import gc
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [2]:
def set_seeds(seed):
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [3]:
# Convert a string that simulates a list to a real list
def convert_string_list(element):
    # Delete [] of the string
    element = element[0:len(element)]
    # Create a list that contains each code as e.g. 'A'
    ATC_list = list(element.split('; '))
    for index, code in enumerate(ATC_list):
        # Delete '' of the code
        ATC_list[index] = code[0:len(code)]
    return ATC_list

In [4]:
def multiplicate_rows(df):
    # Duplicate each compound the number of ATC codes associated to it, copying its SMILES in new rows
    new_rows = []
    
    for _, row in df.iterrows():
        atc_codes = row['ATC Codes']
        atc_codes_list = convert_string_list(atc_codes)
        
        if len(atc_codes_list) > 1:
            for code in atc_codes_list:
                if len(code) == 5:
                    new_row = row.copy()
                    new_row['ATC Codes'] = code
                    new_rows.append(new_row)
        else:
            if len(atc_codes_list[0]) == 5:
                new_rows.append(row)
    
    new_set = pd.DataFrame(new_rows)
    new_set = new_set.reset_index(drop=True)

    return new_set

def extract_descriptors(df):
    """
    Extract molecular descriptors from your dataset.
    You'll need to implement this based on your descriptor source.
    
    Returns FloatTensor of shape (n_molecules, descriptor_dimension)
    """
    descriptors = df.iloc[:, 2:-5].values
    # Convert to numpy for easier handling
    if isinstance(descriptors, torch.Tensor):
        desc_array = descriptors.numpy()
    else:
        desc_array = np.array(descriptors)
    
    # Replace infinite values with NaN first
    desc_array[np.isinf(desc_array)] = np.nan
    
    # Calculate median for each feature (column-wise)
    medians = np.nanmedian(desc_array, axis=0)
    
    # Replace NaN values with corresponding median
    for i in range(desc_array.shape[1]):
        mask = np.isnan(desc_array[:, i])
        desc_array[mask, i] = medians[i]
    return torch.tensor(desc_array, dtype=torch.float32)
    
# Create vocabularies
# Tokenize the data
def source(df):
    source = []
    for compound in df['Neutralized SMILES']:
        # A list containing each SMILES character separated
        source.append(list(compound))
    return source
def target(df):
    target = []
    for codes in df['ATC Codes']:  
        code = convert_string_list(codes) 
        # A list of lists, each one containing each ATC code character separated 
        for c in code:
            list_c = list(c)
            target.append(list_c)
    return target

In [5]:
def f1_calc(output_beam, df, k):
    f1s = []
    for i, preds in enumerate(output_beam):
        ground_truth = convert_string_list(df['ATC Codes'][i])
        binary_predictions = []
        binary_ground_truth = []
        clean_preds = []
        for pred in preds[0:k]:
            p = pred[7:-5]
            clean_preds.append(p)
        set_pred_gt = list(set(clean_preds + ground_truth))
        for code in set_pred_gt:
            if code in clean_preds:
                binary_predictions.append(1)
            else:
                binary_predictions.append(0)
            if code in ground_truth:
                binary_ground_truth.append(1)
            else:
                binary_ground_truth.append(0)    
        f1s.append(metrics.f1_score(binary_ground_truth, binary_predictions))
    return f1s

In [6]:
def metrics_k_test(k, output_beam_test):
    f1sk = []
    precisionsk = []
    recallsk = []
    outputk = []
    for i, out in enumerate(output_beam_test):
        outputk.append(out[0:k])
    for i, preds in enumerate(outputk):
        ground_truth = convert_string_list(test_set['ATC Codes'][i])
        binary_predictions = []
        binary_ground_truth = []
        clean_preds = []
        for pred in preds[0:len(preds)]:
            p = pred[7:-5]
            clean_preds.append(p)
        set_pred_gt = list(set(clean_preds + ground_truth))
        for code in set_pred_gt:
            if code in clean_preds:
                binary_predictions.append(1)
            else:
                binary_predictions.append(0)
            if code in ground_truth:
                binary_ground_truth.append(1)
            else:
                binary_ground_truth.append(0)    
        f1sk.append(metrics.f1_score(binary_ground_truth, binary_predictions))
        precisionsk.append(metrics.precision_score(binary_ground_truth, binary_predictions))
        recallsk.append(metrics.recall_score(binary_ground_truth, binary_predictions))
    average_f1k = sum(f1sk) / len(f1sk)
    average_precisionk = sum(precisionsk) / len(precisionsk)
    average_recallk = sum(recallsk) / len(recallsk)
    return average_precisionk, average_recallk, average_f1k

In [7]:
# def is_invalidATCcode(pred):
#     if not pred.startswith("<START>"):
#         return True
#     if not pred.endswith("<END>"):
#         return True

#     if pred.count("<END>") != 1:
#         return True

#     atccode = pred.replace("<START>", "").replace("<END>", "").strip()

#     patron = r"^[A-Z][0-9]{2}[A-Z]{2}$"

#     if not re.match(patron, atccode):
#         return True
#     return False 

In [8]:
seeds = [42, 123, 47899, 2025, 1, 20, 99, 1020, 345, 78] 
columns = [
    'Seed', 
    'Precision', 'Recall', 'F1',
    'Precision level 1', 'Precision level 2', 'Precision level 3', 'Precision level 4',
    'Recall level 1', 'Recall level 2', 'Recall level 3', 'Recall level 4',
    '#Compounds that have at least one match'
]
predictedk_df = pd.DataFrame()
metrics_df = pd.DataFrame(columns=columns)

for seed in seeds:
    train_set = pd.read_csv(f'../1new_compounds/Datasets/train_set{seed}.csv')
    test_set = pd.read_csv(f'../1new_compounds/Datasets/test_set{seed}.csv') # Test set only used for meta-model testing
    val_set = pd.read_csv(f'../1new_compounds/Datasets/val_set{seed}.csv')

    # Train set for the meta-model
    train_set_meta = pd.concat([train_set, val_set])
    train_set_meta = train_set_meta.reset_index(drop=True)
    train_set_meta = shuffle(train_set_meta, random_state = seed)
    train_set_meta = train_set_meta.reset_index(drop=True)
    
    new_train_set = multiplicate_rows(train_set_meta)
    new_test_set = multiplicate_rows(test_set)
    new_val_set = multiplicate_rows(val_set)
    
    train_descriptors = extract_descriptors(new_train_set)
    train_descriptors2 = extract_descriptors(train_set_meta)
    test_descriptors = extract_descriptors(new_test_set)
    test_descriptors2 = extract_descriptors(test_set)
    val_descriptors = extract_descriptors(new_val_set)
    
    scaler = StandardScaler()
    train_descriptors = torch.tensor(scaler.fit_transform(train_descriptors.numpy()), dtype=torch.float32)
    train_descriptors2 = torch.tensor(scaler.transform(train_descriptors2.numpy()), dtype=torch.float32)
    test_descriptors = torch.tensor(scaler.transform(test_descriptors.numpy()), dtype=torch.float32)
    test_descriptors2 = torch.tensor(scaler.transform(test_descriptors2.numpy()), dtype=torch.float32)
    val_descriptors = torch.tensor(scaler.transform(val_descriptors.numpy()), dtype=torch.float32)
    
    source_train = source(new_train_set)
    # Train set without duplicated compounds
    source_train2 = source(train_set_meta)
    source_test = source(new_test_set)
    source_val = source(new_val_set)
    # Test set without duplicated compounds
    source_test2 = source(test_set)
    
    target_train = target(new_train_set)
    target_test = target(new_test_set)
    target_val = target(new_val_set)
    
    # An Index object represents a mapping from the vocabulary to integers (indices) to feed into the models
    source_index = torch.load(f"../1new_compounds/MMBiLSTM/source_index{seed}.pt", weights_only=False)
    target_index = torch.load(f"../1new_compounds/MMBiLSTM/target_index{seed}.pt", weights_only=False)
    
    # Create tensors
    X_train = source_index.text2tensor(source_train)
    X_train2 = source_index.text2tensor(source_train2)
    y_train = target_index.text2tensor(target_train)    
    X_test = source_index.text2tensor(source_test)
    X_test2 = source_index.text2tensor(source_test2)
    y_test = target_index.text2tensor(target_test)
    X_val = source_index.text2tensor(source_val)
    y_val = target_index.text2tensor(target_val)
    
    if torch.cuda.is_available():
        X_train = X_train.to("cuda")
        X_train2 = X_train2.to("cuda")
        y_train = y_train.to("cuda")
        train_descriptors = train_descriptors.to("cuda") 
        train_descriptors2 = train_descriptors2.to("cuda") 
        test_descriptors = test_descriptors.to("cuda")
        test_descriptors2 = test_descriptors2.to("cuda")
        X_test= X_test.to("cuda")
        y_test = y_test.to("cuda")
        X_test2 = X_test2.to("cuda")
        X_val = X_val.to("cuda")
        y_val = y_val.to("cuda")
        val_descriptors = val_descriptors.to("cuda")
    
    model = pickle.load(open(f"../1new_compounds/MMBiLSTM/modelMultimodalBiLSTM{seed}.pkl", 'rb')) # Trained with train_set and validated with val_set
    model.to("cuda")
    loss, error_rate = model.evaluate(X_train, train_descriptors, y_train, batch_size = 32) 
    torch.cuda.empty_cache()
    b_width = 10
    # predictions, log_probabilities = search_algorithms.multimodal_beam_search(
    #     model, 
    #     X_train2,
    #     train_descriptors2,
    #     predictions = 6, # max length of the predicted sequence
    #     beam_width = b_width,
    #     batch_size = 32, 
    #     progress_bar = 0
    # )
    # output_beam = [target_index.tensor2text(p) for p in predictions]
    predictions1, log_probabilities1 = search_algorithms.multimodal_beam_search(
        model, 
        X_train2[0:854], # Make predictions with test set
        train_descriptors2[0:854],
        predictions = 6, # max length of the predicted sequence
        beam_width = b_width,
        batch_size = 32, 
        progress_bar = 0
    )
    predictions2, log_probabilities2 = search_algorithms.multimodal_beam_search(
        model, 
        X_train2[854:1708], # Make predictions with test set
        train_descriptors2[854:1708],
        predictions = 6, # max length of the predicted sequence
        beam_width = b_width,
        batch_size = 32, 
        progress_bar = 0
    )
    predictions3, log_probabilities3 = search_algorithms.multimodal_beam_search(
        model, 
        X_train2[1708:2561], # Make predictions with test set
        train_descriptors2[1708:2561],
        predictions = 6, # max length of the predicted sequence
        beam_width = b_width,
        batch_size = 32, 
        progress_bar = 0
    )
    output_beam1 = [target_index.tensor2text(p) for p in predictions1]
    output_beam2 = [target_index.tensor2text(p) for p in predictions2]
    output_beam3 = [target_index.tensor2text(p) for p in predictions3]
    output_beam = output_beam1 + output_beam2 + output_beam3

    f1s1 = f1_calc(output_beam, train_set_meta, 1)
    f1s2 = f1_calc(output_beam, train_set_meta, 2)
    f1s3 = f1_calc(output_beam, train_set_meta, 3)
    f1s4 = f1_calc(output_beam, train_set_meta, 4)
    f1s5 = f1_calc(output_beam, train_set_meta, 5)
    f1s6 = f1_calc(output_beam, train_set_meta, 6)
    f1s7 = f1_calc(output_beam, train_set_meta, 7)
    f1s8 = f1_calc(output_beam, train_set_meta, 8)
    f1s9 = f1_calc(output_beam, train_set_meta, 9)
    f1s10 = f1_calc(output_beam, train_set_meta, 10)

    f1 = pd.DataFrame(columns=["1", "2", "3", "4", "5", "6", "7", "8", "9", "10"])
    f1["1"] = f1s1
    f1["2"] = f1s2
    f1["3"] = f1s3
    f1["4"] = f1s4
    f1["5"] = f1s5
    f1["6"] = f1s6
    f1["7"] = f1s7
    f1["8"] = f1s8
    f1["9"] = f1s9
    f1["10"] = f1s10

    col_max_por_fila = f1.idxmax(axis=1)
    filas_todo_cero = (f1 == 0.0).all(axis=1)
    col_max_por_fila[filas_todo_cero] = '1'
    col_max_por_fila.tolist()

    scaler_probs = StandardScaler()
    log_probabilities = torch.cat([log_probabilities1, log_probabilities2, log_probabilities3], axis = 0)
    X = log_probabilities.cpu().numpy()
    X = scaler.fit_transform(X)
    y = np.array(col_max_por_fila, dtype=np.float32)
    print(np.mean(y))
    print(np.std(y))
    K.clear_session()
    gc.collect()

    nn = Sequential([
        (Input(shape = (X.shape[1],))),
        (Dense(64, activation = 'relu')),
        (Dense(32, activation = 'relu')),
        (Dense(1, activation = 'linear'))
    ])
    
    nn.compile(
        loss = keras.losses.Huber(),
        optimizer = keras.optimizers.Adam(learning_rate=1e-4),
        metrics = ['mae']
    )
    
    callback = keras.callbacks.EarlyStopping(monitor='val_loss', patience = 20, restore_best_weights=True, verbose = 1)
    
    history = nn.fit(
        X, 
        y, 
        epochs = 5000, 
        batch_size = 32,
        validation_split = 0.15,
        callbacks = [callback], 
        verbose = 0
    )
    pickle.dump(nn, open(f'meta-model_multimodalbiLSTM{seed}.pkl','wb'))
    loss, error_rate = model.evaluate(X_test, test_descriptors, y_test, batch_size = 32) 
    
    # Generate ATC codes for the test set to evaluate the meta-model
    b_width = 10
    predictions_test, log_probabilities_test = search_algorithms.multimodal_beam_search(
        model, 
        X_test2, # Make predictions with test set
        test_descriptors2,
        predictions = 6, # max length of the predicted sequence
        beam_width = b_width,
        batch_size = 32, 
        progress_bar = 0
    )
    output_beam_test = [target_index.tensor2text(p) for p in predictions_test]
    ## Y_TEST_TRUE

    f1s1_test = f1_calc(output_beam_test, test_set, 1)
    f1s2_test = f1_calc(output_beam_test, test_set, 2)
    f1s3_test = f1_calc(output_beam_test, test_set, 3)
    f1s4_test = f1_calc(output_beam_test, test_set, 4)
    f1s5_test = f1_calc(output_beam_test, test_set, 5)
    f1s6_test = f1_calc(output_beam_test, test_set, 6)
    f1s7_test = f1_calc(output_beam_test, test_set, 7)
    f1s8_test = f1_calc(output_beam_test, test_set, 8)
    f1s9_test = f1_calc(output_beam_test, test_set, 9)
    f1s10_test = f1_calc(output_beam_test, test_set, 10)

    f1_test = pd.DataFrame(columns=["1", "2", "3", "4", "5", "6", "7", "8", "9", "10"])
    f1_test["1"] = f1s1_test
    f1_test["2"] = f1s2_test
    f1_test["3"] = f1s3_test
    f1_test["4"] = f1s4_test
    f1_test["5"] = f1s5_test
    f1_test["6"] = f1s6_test
    f1_test["7"] = f1s7_test
    f1_test["8"] = f1s8_test
    f1_test["9"] = f1s9_test
    f1_test["10"] = f1s10_test

    col_max_por_fila_test = f1_test.idxmax(axis=1)
    filas_todo_cero_test = (f1_test == 0.0).all(axis=1)
    col_max_por_fila_test[filas_todo_cero_test] = '1'
    col_max_por_fila_test.tolist()
    y_test_true = np.array(col_max_por_fila_test, dtype=np.float32)
    X_test_probs = log_probabilities_test.cpu().numpy()
    X_test_probs = scaler.transform(X_test_probs)
    nn.evaluate(X_test_probs, y_test_true)
    
    y_pred = nn.predict(X_test_probs)

    mse = mean_squared_error(y_test_true, y_pred)
    mae = mean_absolute_error(y_test_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test_true, y_pred)
    
    print("MSE:", mse)
    print("MAE:", mae)
    print("RMSE:", rmse)
    print("R2:", r2)
    y_pred_int = y_pred.round().astype(int)
    y_pred_int = np.clip(y_pred_int, 1, 10)
    predictedk_df[f"pred_k{seed}"] = y_pred_int.flatten()
    optimized_output = []
    for i, out in enumerate(output_beam_test):
        # preds_i = []
        # for pred in out:
        #     if len(preds_i) == y_pred_int[i][0]:
        #         break
        #     if not is_invalidATCcode(pred):
        #         preds_i.append(pred)
        # optimized_output.append(preds_i)
        optimized_output.append(out[0:y_pred_int[i][0]])
    f1s = []
    precisions = []
    recalls = []
    for i, preds in enumerate(optimized_output):
        ground_truth = convert_string_list(test_set['ATC Codes'][i])
        binary_predictions = []
        binary_ground_truth = []
        clean_preds = []
        for pred in preds[0:len(preds)]:
            p = pred[7:-5]
            clean_preds.append(p)
        set_pred_gt = list(set(clean_preds + ground_truth))
        for code in set_pred_gt:
            if code in clean_preds:
                binary_predictions.append(1)
            else:
                binary_predictions.append(0)
            if code in ground_truth:
                binary_ground_truth.append(1)
            else:
                binary_ground_truth.append(0)    
        f1s.append(metrics.f1_score(binary_ground_truth, binary_predictions, zero_division=0.0))
        precisions.append(metrics.precision_score(binary_ground_truth, binary_predictions, zero_division=0.0))
        recalls.append(metrics.recall_score(binary_ground_truth, binary_predictions, zero_division=0.0))
    average_f1 = sum(f1s) / len(f1s)
    average_precision = sum(precisions) / len(precisions)
    average_recall = sum(recalls) / len(recalls)
    optimized_output_clean = []
    for i, preds in enumerate(optimized_output):
        interm = []
        for pred in preds:
            clean_pred = pred.replace('<START>', '').replace('<END>', '')
            if len(clean_pred) == 5:
                interm.append(clean_pred)
        optimized_output_clean.append(interm)
    precision_1, precision_2, precision_3, precision_4 = defined_metrics.precision(optimized_output_clean, f'../1new_compounds/Datasets/test_set{seed}.csv', 'ATC Codes')
    recall_1, recall_2, recall_3, recall_4, counter_compound_match = defined_metrics.recall(optimized_output_clean, f'../1new_compounds/Datasets/test_set{seed}.csv', 'ATC Codes')

    metrics_values = {
        'Precision': average_precision, 
        'Recall': average_recall,
        'F1': average_f1,
        'Precision level 1': precision_1,
        'Precision level 2': precision_2,
        'Precision level 3': precision_3,
        'Precision level 4': precision_4,
        'Recall level 1': recall_1,
        'Recall level 2': recall_2,
        'Recall level 3': recall_3,
        'Recall level 4': recall_4,
        '#Compounds that have at least one match': counter_compound_match
    }
    
    row = {
        'Seed': seed,
        **metrics_values
    }
    
    metrics_df = pd.concat([metrics_df, pd.DataFrame([row])], ignore_index=True)

metrics_df.to_csv("meta-model_metrics.csv", index=False)
predictedk_df.to_csv("predictedk.csv", index = False)
print("Mean:", metrics_df.mean(numeric_only=True))
print("Std:", metrics_df.std(numeric_only=True))

1.6923077
1.40723

Epoch 51: early stopping
Restoring model weights from the end of the best epoch: 31.
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 997us/step - loss: 0.7276 - mae: 1.0695
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 
MSE: 3.4270034
MAE: 1.0243964
RMSE: 1.8512167
R2: 0.03860348463058472


C:\Users\trini\AppData\Local\Temp\ipykernel_421556\802263187.py:321: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  metrics_df = pd.concat([metrics_df, pd.DataFrame([row])], ignore_index=True)


1.9714955
1.8072826
Epoch 274: early stopping
Restoring model weights from the end of the best epoch: 254.
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.6836 - mae: 1.0689
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 
MSE: 3.238048
MAE: 1.102788
RMSE: 1.7994577
R2: 0.03919041156768799
1.8789536
1.651201
Epoch 116: early stopping
Restoring model weights from the end of the best epoch: 96.
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.6745 - mae: 1.0349 
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 
MSE: 3.275828
MAE: 1.0405899
RMSE: 1.8099248
R2: 0.016237616539001465
2.1737602
2.0897171
Epoch 100: early stopping
Restoring model weights from the end of the best epoch: 80.
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.8672 - mae: 1.2256
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 
MSE: 4.177136
MAE: 1.1334925
RMSE: 2.0438042
R2: 0.014790713787078857
1.8527919
1.6251036
Epoch 111: early stopping
Restoring model weights from the end of the best epoch: 91.
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - lo

In [9]:
optimized_output_clean

[['L01XX'],
 ['J01DH'],
 ['P01BC', 'P01BF'],
 ['L01AD'],
 ['D07AC', 'S01BA'],
 ['S01AX', 'S01XX'],
 ['J05AR'],
 ['L01XX'],
 ['N05AX', 'L04AX'],
 ['G01AE'],
 ['C03AA'],
 ['A11CC'],
 ['J04AX', 'L01XX'],
 ['J02BD'],
 ['N02AD', 'N06AD'],
 ['N03AX', 'N06BX'],
 ['J05AB', 'L01BB'],
 ['L01AC'],
 ['D07AB', 'S01CA'],
 ['J05AX', 'J05AD'],
 ['N05AX', 'N05AD'],
 ['A03BB', 'R03BB'],
 ['A11CC'],
 ['A16AX', 'G01AX'],
 ['N01AB', 'N01AX'],
 ['R06AA', 'N06AA'],
 ['L01CA'],
 ['L01EA', 'L01EX'],
 ['R06AA'],
 ['G01AE'],
 ['L01XE', 'L01EX', 'L01EK'],
 ['A03AA'],
 ['C02DA', 'D11AX'],
 ['R03AC'],
 ['M02AB'],
 ['C03BA', 'G01AE'],
 ['G01AE'],
 ['J05AC'],
 ['C08CA'],
 ['N06BA', 'S01AA'],
 ['A10AB', 'B03AA'],
 ['L01XX', 'L01AX'],
 ['P01AB'],
 ['N04BB', 'N04BA'],
 ['C07AB'],
 ['N06AX', 'N05AD'],
 ['A10BD'],
 ['L01AX', 'J04BA'],
 ['J01DD', 'J01DC'],
 ['N06BX', 'N07BA'],
 ['N06BA'],
 ['C02CC', 'C02CX'],
 ['N02CA', 'N05CA'],
 ['N02CA', 'N02CD'],
 ['M02AA', 'B01AC'],
 ['C04DA'],
 ['D08AE'],
 ['C09AA'],
 ['A08AA'],
 ['J